# Spaceship Titanic — Kaggle Submission

**Notebook 03 | Final CatBoost baseline submission**

This notebook reconstructs the selected 13-feature CatBoost classifier, fits it on all **8,693 labelled training passengers**, generates predictions for **4,277 unlabelled test passengers**, and validates the final submission CSV.

**Selected configuration:** 500 boosting iterations, depth 6, learning rate 0.05, Logloss, random seed 42. The configuration was chosen during earlier development experiments; this notebook does **not** use Kaggle test labels or estimate a new validation score.

> **Reproducibility:** Place `train.csv`, `test.csv`, and `sample_submission.csv` in the project directory. The notebook can live in that directory or in a `notebooks/` subdirectory. Install `pandas`, `catboost`, and Jupyter in your Python environment.

> **Saved outputs:** Outputs below were captured during the original run and may show an older Windows file path. Rerun the revised notebook to refresh them for your local directory.

## 1. Import dependencies

Import pandas to read and validate CSV files, `Path` to manage paths, and CatBoost to train the classifier.

In [1]:
# ============================================================
# SPACESHIP TITANIC — FIRST KAGGLE SUBMISSION
# ============================================================

# Import pandas for data manipulation.
import pandas as pd

# Import Path for file management.
from pathlib import Path

# Import CatBoost for binary classification.
from catboost import CatBoostClassifier

print("Required libraries imported successfully!")

Required libraries imported successfully!


## 2. Load the official competition files

Load the labelled training set, unlabelled Kaggle test set, and the competition's submission template. The revised path logic works from the project root or an immediate `notebooks/` subfolder, rather than depending on one person's Windows username.

In [2]:
# ============================================================
# STEP 16A: LOAD COMPETITION DATA
# ============================================================

# Define the competition project directory.
# Identify the competition directory from either a project-root
# notebook or a notebook located inside a "notebooks" subfolder.
# Run Jupyter from the project directory (or its notebooks folder).
working_dir = Path.cwd().resolve()
required_files = ("train.csv", "test.csv", "sample_submission.csv")

if all((working_dir / name).is_file() for name in required_files):
    project_root = working_dir
elif all((working_dir.parent / name).is_file() for name in required_files):
    project_root = working_dir.parent
else:
    raise FileNotFoundError(
        "Could not find train.csv, test.csv, and sample_submission.csv "
        "in the current folder or its parent. Run Jupyter from the "
        "Spaceship Titanic project folder."
    )

# Load labelled training observations.
train_df = pd.read_csv(project_root / "train.csv")

# Load Kaggle's unlabelled test observations.
test_df = pd.read_csv(project_root / "test.csv")

# Load the required submission template.
sample_submission = pd.read_csv(
    project_root / "sample_submission.csv"
)

# Display dataset dimensions.
print("Training dataset:", train_df.shape)
print("Test dataset:", test_df.shape)
print("Submission template:", sample_submission.shape)

Training dataset: (8693, 14)
Test dataset: (4277, 13)
Submission template: (4277, 2)


## 3. Extract cabin features consistently

The string `Cabin` has the structure **Deck/Number/Side**. Split it into `CabinDeck` (categorical), `CabinNumber` (numerical), and `CabinSide` (categorical). Apply exactly the same function to training and test data; missing or invalid cabin numbers become `NaN`.

In [3]:
# ============================================================
# STEP 16B: EXTRACT CABIN FEATURES
# ============================================================

def extract_cabin_features(df):
    """
    Extract cabin deck, number, and side
    from the original Cabin variable.
    """

    # Avoid changing the original DataFrame.
    df = df.copy()

    # Split cabin information into three components.
    cabin_parts = df["Cabin"].str.split("/", expand=True)

    # Extract cabin deck.
    df["CabinDeck"] = cabin_parts[0]

    # Extract cabin number and convert it to numeric.
    df["CabinNumber"] = pd.to_numeric(
        cabin_parts[1],
        errors="coerce"
    )

    # Extract cabin side.
    df["CabinSide"] = cabin_parts[2]

    # Remove the original combined Cabin variable.
    df = df.drop(columns=["Cabin"])

    return df


# Apply the same transformation to both datasets.
train_model_df = extract_cabin_features(train_df)
test_model_df = extract_cabin_features(test_df)

print("Cabin feature extraction completed!")

Cabin feature extraction completed!


## 4. Define the same 13 predictors used during development

Use **seven numeric** predictors and **six categorical** predictors, and keep test feature columns in precisely the same order as training. `Transported` is the target, not a predictor. We do not use `PassengerId` or passenger names as model inputs.

In [4]:
# ============================================================
# STEP 16C: SELECT MODEL PREDICTORS
# ============================================================

# Define numerical predictors.
numerical_features = [
    "Age",
    "RoomService",
    "FoodCourt",
    "ShoppingMall",
    "Spa",
    "VRDeck",
    "CabinNumber"
]

# Define categorical predictors.
categorical_features = [
    "HomePlanet",
    "CryoSleep",
    "Destination",
    "VIP",
    "CabinDeck",
    "CabinSide"
]

# Combine all selected predictors.
selected_features = numerical_features + categorical_features

# Create the full labelled training predictors.
X_full = train_model_df[selected_features].copy()

# Create the training target.
y_full = train_model_df["Transported"].copy()

# Create the Kaggle test predictors.
X_kaggle = test_model_df[selected_features].copy()

print("Training features:", X_full.shape)
print("Kaggle test features:", X_kaggle.shape)

Training features: (8693, 13)
Kaggle test features: (4277, 13)


## 5. Prepare categorical values for CatBoost

Convert categorical columns to strings and use the explicit label `"Missing"` for absent categories. CatBoost can handle missing **numeric** values directly; they are intentionally left as `NaN`. Unlike the scikit-learn Logistic Regression pipeline, this model does not require one-hot encoding or numerical scaling.

In [5]:
# ============================================================
# STEP 16D: PREPARE CATEGORICAL FEATURES
# ============================================================

# Convert categorical columns into string values.
# Represent missing categories explicitly as "Missing".
for column in categorical_features:

    X_full[column] = (
        X_full[column]
        .astype("string")
        .fillna("Missing")
        .astype(str)
    )

    X_kaggle[column] = (
        X_kaggle[column]
        .astype("string")
        .fillna("Missing")
        .astype(str)
    )

# Confirm that categorical predictors contain no missing markers.
print(
    "Missing categorical values in full training data:",
    X_full[categorical_features].isna().sum().sum()
)

print(
    "Missing categorical values in Kaggle test data:",
    X_kaggle[categorical_features].isna().sum().sum()
)

Missing categorical values in full training data: 0
Missing categorical values in Kaggle test data: 0


## 6. Train the final classifier on all labelled passengers

Once development experiments are complete, fit a fresh model on the **full labelled training dataset** to make the competition submission. This is different from the earlier 80/20 holdout and five-fold CV procedures used to estimate performance.

In [6]:
# ============================================================
# STEP 16E: TRAIN FINAL CATBOOST MODEL
# ============================================================

# Create a new CatBoost model.
final_catboost_model = CatBoostClassifier(
    iterations=500,
    depth=6,
    learning_rate=0.05,
    loss_function="Logloss",
    random_seed=42,
    verbose=False
)

# Train on all available labelled passengers.
final_catboost_model.fit(
    X_full,
    y_full,
    cat_features=categorical_features
)

print("Final CatBoost model trained successfully!")

Final CatBoost model trained successfully!


## 7. Generate predictions for Kaggle's unlabelled test set

Predictions are converted to Boolean values (`True` or `False`) to match the competition format. These are model predictions—not observed labels.

In [7]:
# ============================================================
# STEP 16F: GENERATE KAGGLE TEST PREDICTIONS
# ============================================================

# Generate predictions for the Kaggle test dataset.
kaggle_predictions = (
    final_catboost_model.predict(X_kaggle)
    .ravel()
    .astype(bool)
)

# Display the number of predictions.
print("Total predictions:", len(kaggle_predictions))

# Display the first ten predictions.
print("First ten predictions:")
print(kaggle_predictions[:10])

Total predictions: 4277
First ten predictions:
[ True False  True  True  True  True  True  True  True  True]


## 8. Build and save the submission CSV

Populate the official submission template to preserve passenger order and the required column names. Assertions verify structure, count, alignment, and absence of missing predictions. The CSV is saved under `submissions/catboost_baseline_submission.csv`.

In [8]:
# ============================================================
# STEP 16G: CREATE SUBMISSION FILE
# ============================================================

# Create a copy of the official Kaggle submission template.
submission = sample_submission.copy()

# Insert our model predictions into the Transported column.
submission["Transported"] = kaggle_predictions

# Confirm that each prediction corresponds to the correct passenger.
assert (
    submission["PassengerId"].tolist()
    == test_df["PassengerId"].tolist()
), "Passenger IDs do not match the Kaggle test dataset."

# Confirm that the submission contains the expected columns.
assert submission.columns.tolist() == [
    "PassengerId",
    "Transported"
], "Submission columns are incorrect."

# Confirm that the number of predictions is correct.
assert len(submission) == len(test_df), (
    "Submission row count does not match the test dataset."
)

# Check for missing predictions.
assert submission["Transported"].notna().all(), (
    "Submission contains missing predictions."
)

# Create the submissions directory if it does not exist.
submission_dir = project_root / "submissions"
submission_dir.mkdir(exist_ok=True)

# Define the output file path.
submission_path = (
    submission_dir / "catboost_baseline_submission.csv"
)

# Save the submission without a pandas index.
submission.to_csv(
    submission_path,
    index=False
)

# Display the saved file location.
print("Submission file saved successfully!")
print("File location:", submission_path)

# Preview the first five predictions.
display(submission.head())

Submission file saved successfully!
File location: C:\Users\A\Desktop\sample_project_1\spaceship-titanic\submissions\catboost_baseline_submission.csv


,PassengerId,Transported
0,0013_01,True
1,0018_01,False
2,0019_01,True
3,0021_01,True
4,0023_01,True


## 9. Verify the CSV as saved on disk

Read the file back and verify its **4,277 rows**, **two columns**, unique passenger IDs, valid Boolean predictions, and matching passenger order. This final check catches issues that may occur between DataFrame creation and CSV export.

**Recorded Kaggle public leaderboard accuracy:** **0.80383 (80.383%)**. This score came from the separate Kaggle submission, not from the assertions below; public leaderboard performance is not an independent estimate of the hidden private leaderboard score.

In [9]:
# ============================================================
# STEP 16H: VERIFY THE SAVED KAGGLE SUBMISSION
# ============================================================

# Read the saved CSV file from disk.
# This checks the actual file that will be uploaded to Kaggle.
submission_check = pd.read_csv(submission_path)

# Confirm that the CSV has exactly 4,277 predictions.
assert submission_check.shape == (4277, 2), (
    "Incorrect submission dimensions."
)

# Confirm that Kaggle's required columns are present.
assert submission_check.columns.tolist() == [
    "PassengerId",
    "Transported"
], "Incorrect submission column names."

# Confirm that passenger IDs match the competition test data.
assert submission_check["PassengerId"].tolist() == (
    test_df["PassengerId"].tolist()
), "Passenger IDs do not match."

# Confirm that passenger IDs are unique.
assert submission_check["PassengerId"].is_unique, (
    "Duplicate passenger IDs detected."
)

# Confirm that no predictions are missing.
assert submission_check["Transported"].notna().all(), (
    "Missing predictions detected."
)

# Confirm that predictions contain only True or False.
assert submission_check["Transported"].isin(
    [True, False]
).all(), "Invalid prediction values detected."

# Display the final verification results.
print("SUBMISSION VERIFICATION")
print("-" * 45)

print("File:", submission_path.name)
print("Rows:", submission_check.shape[0])
print("Columns:", submission_check.shape[1])
print("Missing values:", submission_check.isna().sum().sum())
print("Duplicate IDs:", submission_check["PassengerId"].duplicated().sum())

print("\nPrediction distribution:")
print(submission_check["Transported"].value_counts())

print("\nAll submission checks passed successfully!")

# Display the first five rows.
display(submission_check.head())

SUBMISSION VERIFICATION
---------------------------------------------
File: catboost_baseline_submission.csv
Rows: 4277
Columns: 2
Missing values: 0
Duplicate IDs: 0

Prediction distribution:
Transported
True     2187
False    2090
Name: count, dtype: int64

All submission checks passed successfully!


,PassengerId,Transported
0,0013_01,True
1,0018_01,False
2,0019_01,True
3,0021_01,True
4,0023_01,True


## Conclusions and limitations

- This notebook creates and validates a structurally correct Kaggle submission from the selected CatBoost baseline.
- Model-development CV accuracy was approximately **81.12%**; group-aware CV averaged **80.85%**. These are results of earlier development experiments, not new scores calculated here.
- The public leaderboard result for this submission was **80.383%**, as recorded from Kaggle.
- Kaggle's test labels are not available here, so the notebook **cannot verify prediction accuracy** on the competition test set.
- Successful saved-output checks do not replace a fresh **Restart Kernel → Run All** reproducibility test after the data files are available locally.
